# Strands Decider 2B on a free Colab GPU
**Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.
Needs `MyDrive/fin-decider/fin-decider-colab.tar.gz` (built on the Mac with `colab/package.sh`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1. Mount Drive, unpack the package, check out the exact commit, install the pipeline.
from google.colab import drive
drive.mount('/content/drive')
import json, os, subprocess, time
DRIVE = '/content/drive/MyDrive/fin-decider'
OUT = f'{DRIVE}/results'
os.makedirs(OUT, exist_ok=True)
!rm -rf /content/pkg && mkdir -p /content/pkg && tar -xzf {DRIVE}/fin-decider-colab.tar.gz -C /content/pkg
BUNDLE = json.load(open('/content/pkg/bundle.json')); print(BUNDLE)
!rm -rf /content/fin-decider && git clone -q /content/pkg/repo.bundle /content/fin-decider -b {BUNDLE['branch']}
%cd /content/fin-decider
!git checkout -q {BUNDLE['commit']} && git log --oneline -1
!cp -r /content/pkg/corpus_native . && (test -d /content/pkg/corpus && cp -r /content/pkg/corpus . || echo "no Bloomberg corpus in package (Reuters only)")

MODEL = 'strands'

In [ ]:
# 2. Model package and pinned weights (the LoRA adapter; its Qwen3.5-2B base is fetched by the package).
!pip install -q strands-decider==0.2.0
from huggingface_hub import snapshot_download
CKPT = snapshot_download('StrandsAgents/strands-decider-2B-hobson-v19', revision='bb282d786bc251fd4e3068de3ada9ddbb38127cd')
print(CKPT)

In [ ]:
# 3. Install the pipeline after the model package, so pip keeps the model's pinned dependencies.
!pip install -q -e . && classify --help | head -3
!pip freeze > {OUT}/pip_freeze_{MODEL}.txt; nvidia-smi > {OUT}/gpu_{MODEL}.txt

In [ ]:
# 4. Start the server in the background and wait until it answers a test question.
import requests
log = open(f'{OUT}/strands_server.log', 'a')
server = subprocess.Popen(['strands-decider', 'serve', CKPT, '--port', '8000', '--device', 'cuda',
                           '--model-name', 'strands-decider-2B-hobson-v19', '--strict-window'], stdout=log, stderr=subprocess.STDOUT)
probe = {'state': 'Bank of America arranged a 500 million dollar syndicated loan for a utility.',
         'questions': {'q1': {'type': 'noul', 'instructions': 'Does the article describe a bank lending money?'}}}
r = None
for _ in range(90):  # up to 15 minutes for the first model download
    try:
        r = requests.post('http://127.0.0.1:8000/v1/systemone', json=probe, timeout=120)
        if r.ok:
            break
    except requests.RequestException:
        pass
    if server.poll() is not None:
        raise RuntimeError('server exited; see strands_server.log on Drive')
    time.sleep(10)
print(r.status_code if r is not None else 'no response'); print(json.dumps(r.json(), indent=1)[:1500])
MODEL_NAME = r.json().get('model')
json.dump(r.json(), open(f'{OUT}/strands_probe.json', 'w'), indent=1)
print('model name reported by the server:', MODEL_NAME)
over = (f'--set models.strands.expected_model_returned={MODEL_NAME}' if MODEL_NAME
        else '--set models.strands.allow_returned_model_mismatch=true')
print(over)

## Run strands on the Reuters top-10 test set (2,533 stories, v1 questions)
Comparable with the Jev, d1 and Clef v1 runs. Results and logs are written to Drive, and `--resume` continues after a disconnect: just rerun the setup cells, the server cell and this cell.

In [ ]:
!classify run -c configs/native_reuters_top10_local.yaml -m strands --no-dev-eval --resume \
   --output-dir {OUT}/classification_results_native --reports-dir {OUT}/reports {over} 2>&1 | tee -a {OUT}/strands_reuters.log

## Optional: strands on the first 1,000 Bloomberg articles (G-SIB ontology)
The same 1,000 articles as the Jev, d1, Clef and Laya engineering runs; includes the synthetic dev-set check. Needs the full package (not `--reuters-only`).

In [ ]:
!classify run -c configs/classification.yaml -m strands --limit 1000 --resume --set models.strands.enabled=true \
   --output-dir {OUT}/classification_results --reports-dir {OUT}/reports {over} 2>&1 | tee -a {OUT}/strands_bloomberg1000.log

## Done
Download `MyDrive/fin-decider/results/` and tell Claude where it is; the runs are merged into the repo's results folders and scored there.